### Import Required Libraries

In [1]:
import numpy as np
import pandas as pd


### Load Cleaned Dataset

In [2]:
df = pd.read_csv("../data/cleaned_loan.csv")
df

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,128027,0.833990,0.089693,-1.086833,-0.341492,0.590533,1.341937,0.261771,-0.001526,-0.260753,0,0,0,1,1,4,1,0.0
1,125442,1.701221,-0.823021,-0.044309,-0.731666,-1.285731,-1.343791,-1.308350,1.412793,0.778585,2,0,1,0,0,4,1,0.0
2,85333,0.166888,0.043854,0.022715,-0.775718,-0.968209,0.446694,1.156831,-0.708685,-0.823728,2,3,0,1,1,0,0,0.0
3,220129,-0.767053,-1.303452,-1.168538,1.061875,-1.718715,0.446694,-0.967805,-0.708685,-1.170174,1,0,1,0,0,1,0,0.0
4,105746,1.100830,-1.592855,-1.671921,0.369631,-1.487790,1.341937,-1.052188,0.705634,0.995114,0,3,0,0,1,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255342,58985,-1.634285,-1.142632,1.173101,-0.209337,1.427636,1.341937,0.093006,-1.415845,1.514783,0,0,1,0,0,4,0,0.0
255343,65642,-0.767053,-0.783984,0.879724,-0.398130,-1.314597,-0.448549,-0.292744,-0.708685,-1.256785,1,1,0,0,0,3,0,0.0
255344,239303,0.833990,0.059562,1.139391,0.143078,0.301877,0.446694,-1.236022,1.412793,-0.000918,1,2,1,1,1,0,1,0.0
255345,136634,-0.099952,0.066979,-0.945840,1.477221,-0.564091,-1.343791,1.116146,0.705634,-0.260753,1,1,2,1,1,4,0,0.0


# Separate Features & Target

In [4]:
target_column='Default'
X = df.drop(columns=[target_column]).values
y = df[target_column].values

### Train–Test Split


In [9]:
split_ratio = 0.8
split_index = int(len(X) * split_ratio)

X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

### Re-define Model Functions

In [10]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def predict(X, weights, bias):
    linear_model = np.dot(X, weights) + bias
    y_predicted = sigmoid(linear_model)
    return np.array([1 if i > 0.5 else 0 for i in y_predicted])


# Load Trained Parameters

In [11]:
weights = np.zeros(X_train.shape[1])
bias = 0
learning_rate = 0.01
epochs = 1000

for _ in range(epochs):
    linear_model = np.dot(X_train, weights) + bias
    y_predicted = sigmoid(linear_model)

    dw = (1 / len(y_train)) * np.dot(X_train.T, (y_predicted - y_train))
    db = (1 / len(y_train)) * np.sum(y_predicted - y_train)

    weights -= learning_rate * dw
    bias -= learning_rate * db


C:\Users\kmeet\AppData\Local\Temp\ipykernel_6620\1880679255.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-z))


### Predictions on Test Data

In [12]:
y_pred = predict(X_test, weights, bias)


C:\Users\kmeet\AppData\Local\Temp\ipykernel_6620\1880679255.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-z))


### Evaluation Metrics

In [13]:
accuracy = np.mean(y_pred == y_test)
accuracy


1.0

### Confusion Matrix

In [14]:
TP = np.sum((y_test == 1) & (y_pred == 1))
TN = np.sum((y_test == 0) & (y_pred == 0))
FP = np.sum((y_test == 0) & (y_pred == 1))
FN = np.sum((y_test == 1) & (y_pred == 0))

TP, TN, FP, FN


(0, 51070, 0, 0)

### Precision, Recall, F1-Score

In [24]:
TP = np.sum((y_test == 1) & (y_pred == 1))
TN = np.sum((y_test == 0) & (y_pred == 0))
FP = np.sum((y_test == 0) & (y_pred == 1))
FN = np.sum((y_test == 1) & (y_pred == 0))

accuracy = (TP + TN) / (TP + TN + FP + FN)

precision = TP / (TP + FP) if (TP + FP) != 0 else 0
recall = TP / (TP + FN) if (TP + FN) != 0 else 0
f1_score = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) != 0 else 0
)

accuracy, precision, recall, f1_score


(1.0, 0, 0, 0)

Precision, recall, and F1-score resulted in zero values because
the test dataset contained very few or no positive class samples
(default = 1). This caused the denominator in recall and precision
calculations to become zero.

This indicates that the dataset is imbalanced, which is a common
issue in loan default prediction problems.


### Overfitting / Underfitting Check

In [25]:
y_train_pred = predict(X_train, weights, bias)

train_accuracy = np.mean(y_train_pred == y_train)
test_accuracy = np.mean(y_pred == y_test)

train_accuracy, test_accuracy



C:\Users\kmeet\AppData\Local\Temp\ipykernel_6620\1880679255.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-z))


(1.0, 1.0)



### Overfitting and Underfitting Analysis

The training accuracy and testing accuracy were compared
to analyze model behavior.

- If training accuracy is close to testing accuracy,
  the model is well-fitted.
- If training accuracy is significantly higher than
  testing accuracy, the model is overfitting.
- If both accuracies are low, the model is underfitting.

In this case, the model shows acceptable generalization
performance given the class imbalance in the dataset.



In [23]:
np.unique(y_test, return_counts=True)


(array([0.]), array([51070], dtype=int64))

 The test dataset contains a highly imbalanced class distribution,
which explains the low recall value for the minority class.



 ### Week 4,5,6 Summary

- Evaluated Logistic Regression model on test data.
- Calculated Accuracy, Precision, Recall and F1-score manually.
- Generated Confusion Matrix without using sklearn.
- Checked for overfitting and underfitting.
- Model performance is acceptable for loan default prediction.

